# Customer Shopping Behaviour Analysis

**Tools:** Python · Pandas · PostgreSQL · Power BI

## Business problem
How can a retail company use customer shopping data to understand purchasing behaviour, identify purchase-history segments, evaluate promotional activity and improve product and customer-engagement decisions?

## Dataset
- 3,900 records
- 18 source columns
- 37 missing values in `review_rating`

This notebook documents data understanding, cleaning, feature engineering and optional PostgreSQL loading. SQL business questions are maintained separately in `sql/customer_behavior_analysis.sql`.

## 1. Import libraries and configure paths

In [2]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "customer_shopping_behavior.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load the dataset

In [3]:
df = pd.read_csv("D:\Selected_Resume\Retail  E-commerce Data Analyst\customer_shopping_behavior.csv")
df.head()

<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\N Nithish\AppData\Local\Temp\ipykernel_23604\2060100764.py:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  df = pd.read_csv("D:\Selected_Resume\Retail  E-commerce Data Analyst\customer_shopping_behavior.csv")


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


## 3. Initial data understanding

In [4]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.info())
display(df.describe(include="all").T)

Rows: 3,900
Columns: 18
<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         390

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Customer ID,3900.0,NaN,NaN,NaN,1950.5,1125.977353,1.0,975.75,1950.5,2925.25,3900.0
Age,3900.0,NaN,NaN,NaN,44.068462,15.207589,18.0,31.0,44.0,57.0,70.0
Gender,3900,2,Male,2652,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Item Purchased,3900,25,Blouse,171,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,3900,4,Clothing,1737,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Purchase Amount (USD),3900.0,NaN,NaN,NaN,59.764359,23.685392,20.0,39.0,60.0,81.0,100.0
Location,3900,50,Montana,96,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Size,3900,4,M,1755,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Color,3900,25,Olive,177,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Season,3900,4,Spring,999,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
missing_summary = (
    df.isna()
      .sum()
      .rename("missing_values")
      .to_frame()
      .query("missing_values > 0")
)
missing_summary

,missing_values
Review Rating,37


In [6]:
duplicate_rows = df.duplicated().sum()
duplicate_customer_ids = df["Customer ID"].duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")
print(f"Repeated Customer IDs: {duplicate_customer_ids}")

Duplicate rows: 0
Repeated Customer IDs: 0


## 4. Standardize column names

Convert column names to lowercase snake_case and simplify `purchase_amount_usd` to `purchase_amount`.

In [7]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

df = df.rename(columns={"purchase_amount_usd": "purchase_amount"})
df.columns.tolist()

['customer_id',
 'age',
 'gender',
 'item_purchased',
 'category',
 'purchase_amount',
 'location',
 'size',
 'color',
 'season',
 'review_rating',
 'subscription_status',
 'shipping_type',
 'discount_applied',
 'promo_code_used',
 'previous_purchases',
 'payment_method',
 'frequency_of_purchases']

## 5. Handle missing review ratings

The 37 missing ratings are imputed using the median rating within each product category. This preserves category-level differences and avoids replacing missing ratings with zero.

The imputation should be documented in the project because it changes the observed data.

In [8]:
missing_before = df["review_rating"].isna().sum()

df["review_rating"] = (
    df.groupby("category")["review_rating"]
      .transform(lambda values: values.fillna(values.median()))
)

missing_after = df["review_rating"].isna().sum()
print(f"Missing ratings before: {missing_before}")
print(f"Missing ratings after: {missing_after}")

Missing ratings before: 37
Missing ratings after: 0


## 6. Create age groups

The original analysis uses quartile-based bins (`qcut`). Therefore, these labels represent relative age groups in this dataset, not fixed demographic definitions that will necessarily match another population.

In [9]:
age_labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]

df["age_group"] = pd.qcut(
    df["age"],
    q=4,
    labels=age_labels
)

df[["age", "age_group"]].head()

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged


## 7. Map purchase frequency to approximate day intervals

These values are representative intervals derived from category labels. They are not measured gaps between dated purchases because the dataset does not contain transaction dates.

In [10]:
frequency_mapping = {
    "Fortnightly": 14,
    "Weekly": 7,
    "Monthly": 30,
    "Quarterly": 90,
    "Bi-Weekly": 14,
    "Annually": 365,
    "Every 3 Months": 90,
}

df["purchase_frequency_days"] = (
    df["frequency_of_purchases"].map(frequency_mapping)
)

unmapped_frequency_values = (
    df.loc[df["purchase_frequency_days"].isna(), "frequency_of_purchases"]
      .dropna()
      .unique()
)

print("Unmapped frequency labels:", unmapped_frequency_values)

Unmapped frequency labels: <StringArray>
[]
Length: 0, dtype: str


## 8. Check promotion-field redundancy

In the supplied dataset, `discount_applied` and `promo_code_used` were found to match row-for-row. This is a dataset-specific observation, not a general rule that the fields mean the same thing. Keep the check in the notebook so the decision is reproducible.

In [11]:
promotion_fields_match = df["discount_applied"].equals(df["promo_code_used"])
print("Promotion fields match exactly:", promotion_fields_match)

if promotion_fields_match:
    df = df.drop(columns=["promo_code_used"])
else:
    print("Keeping both fields because they contain different information.")

Promotion fields match exactly: True


## 9. Validate the cleaned dataset

In [12]:
print("Cleaned shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_values"))

assert df["purchase_amount"].ge(0).all(), "Negative purchase amount found."
assert df["review_rating"].dropna().between(0, 5).all(), "Review rating outside expected range."

Cleaned shape: (3900, 19)
Duplicate rows: 0
Missing values by column:


,missing_values
customer_id,0
age,0
gender,0
item_purchased,0
category,0
purchase_amount,0
location,0
size,0
color,0
season,0


In [13]:
display(df.head())
display(
    df[["purchase_amount", "age", "review_rating", "previous_purchases",
        "purchase_frequency_days"]].describe().T
)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365


,count,mean,std,min,25%,50%,75%,max
purchase_amount,3900.0,59.764359,23.685392,20.0,39.0,60.0,81.0,100.0
age,3900.0,44.068462,15.207589,18.0,31.0,44.0,57.0,70.0
review_rating,3900.0,3.750051,0.713590,2.5,3.1,3.8,4.4,5.0
previous_purchases,3900.0,25.351538,14.447125,1.0,13.0,25.0,38.0,50.0
purchase_frequency_days,3900.0,89.133077,119.037566,7.0,14.0,30.0,90.0,365.0


## 10. Export the cleaned dataset

The processed CSV is written to `data/processed/`. The repository `.gitignore` excludes data files by default; decide separately whether the dataset license permits sharing.

In [15]:
cleaned_path = PROCESSED_DIR / "D:\Selected_Resume\Retail  E-commerce Data Analyst\customer_shopping_behavior.csv"
df.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved to: {cleaned_path}")

Cleaned dataset saved to: D:\Selected_Resume\Retail  E-commerce Data Analyst\customer_shopping_behavior.csv


<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\N Nithish\AppData\Local\Temp\ipykernel_23604\4255598031.py:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  cleaned_path = PROCESSED_DIR / "D:\Selected_Resume\Retail  E-commerce Data Analyst\customer_shopping_behavior.csv"


## 11. Analysis handoff

Run the SQL queries in `sql/customer_behavior_analysis.sql` against the `customer` table. Use the results to build the Power BI report.

### Recommended Power BI measures
- Total Revenue
- Purchase Records
- Unique Customers
- Average Purchase Amount
- Average Review Rating
- Subscription Rate
- Discount Utilization Rate
